In [1]:
!mkdir build && cd build && cmake .. && make

mkdir: build: File exists


In [1]:
import sys
import os
import yaml
import numpy as np # fix 

# --- 1. DYNAMIC PATH SETUP ---
# Look for the .so file in the build directory
build_path = os.path.abspath("./build")
sys.path.append(build_path)

try:
    # This must match the name in your PYBIND11_MODULE(cpp_tbcc_decoder, m)
    import cpp_tbcc_decoder
    print("Successfully imported cpp_tbcc_decoder")
except ImportError as e:
    print(f"Error: Could not import the module.")
    print(f"Details: {e}")
    print(f"Looking in: {build_path}")
    print(f"Files found in build: {os.listdir(build_path) if os.path.exists(build_path) else 'Folder not found'}")
    sys.exit(1)

Successfully imported cpp_tbcc_decoder


In [17]:
def octal_to_binary_list(octal_str):
    # Convert octal string to integer, then to a binary string
    # '0b' prefix is sliced off using [2:]
    binary_str = bin(int(octal_str, 8))[2:]

    # Convert the string '101' into a list of integers [1, 0, 1]
    return [int(bit) for bit in binary_str]


def bin2dec(binary):
    """
    Note: bin_list is intentionally not reversed to read the number as 'right-msb'.
    It can be "reversed" if the next states is intended to be read as 'left-msb'.
    """
    return [
        int(sum(val * (2**idx) for idx, val in enumerate(bin_list)))
        for bin_list in binary
    ]

# remove a w d, use dst states and outputs
# may need to concat dst_0 and dst_1
def setup_A_W_D(code_config):

    nu = code_config["tbcc_config"]["V"]
    m = code_config["bch_config"]["M"]
    K = code_config["bch_config"]["K"]
    crc = code_config["bch_config"]["polynomial"]
    num_concat_memory = nu + m
    num_total_states = 2 ** (num_concat_memory)
    num_valid_starting_states = 2 ** (code_config["tbcc_config"]["V"])
    states = np.arange(0, num_total_states, dtype=np.int32)
    num_trellis_stages = code_config["bch_config"]["K"] + code_config["bch_config"]["M"]
    cwd_max_weight = 2 * num_trellis_stages

    p1 = np.flip(octal_to_binary_list(code_config["tbcc_config"]["gen_poly_1"]))
    p2 = np.flip(octal_to_binary_list(code_config["tbcc_config"]["gen_poly_2"]))
    p_crc = np.flip([int(bit) for bit in code_config["bch_config"]["polynomial"]])

    # convolution between gen_poly with crc
    poly1 = np.mod(np.convolve(p1, p_crc, mode="full"), 2)
    poly2 = np.mod(np.convolve(p2, p_crc, mode="full"), 2)

    # states
    states_str = [np.binary_repr(s, width=num_concat_memory) for s in states]
    flipped_states = [s[::-1] for s in states_str]
    states_matrix = np.array([[int(bit) for bit in s] for s in flipped_states])

    # input, dst states
    v_zeros = np.zeros(shape=(num_total_states, 1))
    v_ones = np.ones(shape=(num_total_states, 1))
    input_0 = np.hstack((v_zeros, states_matrix))
    input_1 = np.hstack((v_ones, states_matrix))
    dst_0 = np.array(bin2dec(input_0[:, :num_concat_memory]))
    dst_1 = np.array(bin2dec(input_1[:, :num_concat_memory]))

    # output
    out0 = np.mod(np.matmul(input_0, np.transpose(np.vstack((poly1, poly2)))), 2).astype(np.int32)
    out1 = np.mod(np.matmul(input_1, np.transpose(np.vstack((poly1, poly2)))), 2).astype(np.int32)

    return dst_0, dst_1, out0, out1

In [26]:
info = cpp_tbcc_decoder.CodeInformation(1, 2, 8, 0, 0, 15, [667, 753])

code = {
  'bch_config': {
    'K': 15,
    'N': 15,
    'M': 0,
    'polynomial': "1"
  },
  'tbcc_config': {
    'K': 15,
    'N': 30,
    'V': 8,
    'gen_poly_1': "667",
    'gen_poly_2': "753"
  }
}
dst_0, dst_1, out0, out1 = setup_A_W_D(code)
dst_ref = np.stack([dst_0.T, dst_1.T], axis=1)
print("dst_ref shape: ", dst_ref.shape)

out0_dec = [int("".join(map(str, bits)), 2) for bits in out0]
out1_dec = [int("".join(map(str, bits)), 2) for bits in out1]
out_ref = np.stack([out0_dec, out1_dec], axis=0).T
print("out_ref shape: ", out_ref.shape)

trellis = cpp_tbcc_decoder.FeedForwardTrellis(info)

# DUT
nextStates = trellis.getNextStates()
output = trellis.getOutputs()

if np.all(out_ref == output):
  print("out same!")
if np.all(dst_ref == nextStates):
  print("dst same!")

dst_ref shape:  (256, 2)
out_ref shape:  (256, 2)
out same!
dst same!
